# LENSIQ Gate: train a Thai licence-plate DETECTOR (YOLOX-nano, Apache-2.0)

Run this on **Google Colab with a GPU runtime** (Runtime > Change runtime type > GPU). It trains a one-class (`plate`)
YOLOX-nano detector on Thai plate datasets **that you downloaded by hand** and checked yourself (see
`docs/PLATE-DATASETS.md`), exports ONNX for onnxruntime-web (WASM) and writes `MODEL_CARD.md` with the dataset
attributions.

**Licence rules (do not change):**
- Trainer: **Megvii YOLOX** (Apache-2.0), pinned below. **Never** Ultralytics (AGPL-3.0) and **never** the
  open-image-models / WongKinYiu YOLOv9 weights (GPL-derived, unclear data) — not as a start point, not to auto-label.
- Start weights: the official YOLOX-nano COCO checkpoint (Apache-2.0, release 0.1.1rc0).
- Data: only datasets whose licence you verified on the page **and** in the export README (CC BY 4.0 / public domain /
  your own pilot photos with PDPA consent). Never NC / ND data. Images are never shipped; only the weights and the card.

Steps: 1 setup · 2 data in · 3 merge + dedupe + one class · 4 split by source · 5 conditions CSV · 6 YOLOX exp ·
7 train · 8 export ONNX (+ optional int8) · 9 evaluate (mAP + per-condition recall) · 10 model card · 11 download.

**Smoke-run log (fill in):** LENSIQ could not run this notebook (no GPU / torch in the build environment). Before a full
run, do one smoke run: steps 1, 6 and 7 with `EPOCHS = 1` on a few images, then note here the date, Colab Python /
torch / CUDA versions and whether it finished. Known Colab issues are handled in step 1 (`--no-build-isolation`,
onnx-simplifier skipped, `TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD` for PyTorch ≥ 2.6).

In [ ]:
# 1. SETUP — GPU check, pinned YOLOX (Apache-2.0), COCO YOLOX-nano start weights (Apache-2.0)
import os, subprocess, sys, json, glob, shutil, hashlib, random, csv, math, time
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU - switch the Colab runtime to GPU")
YOLOX_TAG = "0.3.0"          # pinned release tag; the exact commit hash is printed below and written into MODEL_CARD.md
WORK = "/content/lensiq-plate"
os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
if not os.path.isdir("YOLOX"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", YOLOX_TAG, "https://github.com/Megvii-BaseDetection/YOLOX.git"], check=True)
YOLOX_COMMIT = subprocess.run(["git", "-C", "YOLOX", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("YOLOX", YOLOX_TAG, YOLOX_COMMIT)
# Install notes (not yet smoke-run on Colab by LENSIQ: no GPU here; record your run below):
# - YOLOX's setup.py imports torch, so pip's build isolation fails with "No module named torch": keep Colab's preinstalled
#   torch / torchvision and install YOLOX with --no-build-isolation --no-deps.
# - YOLOX 0.3.0 requirements pin onnx-simplifier==0.4.10, which may not build on Colab's Python 3.11 / 3.12: install the
#   other requirements unpinned and skip onnx-simplifier (export_onnx.py imports it only without --no-onnxsim, which
#   step 8 passes).
req = [l.strip() for l in open("YOLOX/requirements.txt") if l.strip() and not l.startswith("#")]
req = [r for r in req if not r.split(">")[0].split("=")[0].split("<")[0].strip().lower() in ("torch", "torchvision", "onnx-simplifier", "onnx_simplifier", "onnxsim")]
req = [r.split("=")[0].split(">")[0].split("<")[0].strip() for r in req]   # unpinned: Colab's Python needs current wheels
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *req], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-v", "--no-build-isolation", "--no-deps", "-e", "YOLOX"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "pycocotools", "pillow", "numpy"], check=True)
# PyTorch >= 2.6 defaults torch.load(weights_only=True); YOLOX 0.3.0 loads full checkpoints (the official COCO start weights
# and our own ckpts, both trusted) -> allow full loading for the YOLOX subprocesses started below (they inherit os.environ)
import torch
print("torch", torch.__version__)
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
if not os.path.exists("yolox_nano.pth"):
    subprocess.run(["wget", "-q", "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth"], check=True)
print("ready")

## 2. Put the datasets in

Make **one folder per dataset source** inside `RAW` (Drive or upload), each holding the export exactly as downloaded
(Roboflow "COCO JSON", "YOLO" (txt) or "Pascal VOC" export, or Kaggle VOC xml). Add a `SOURCE.json` to each folder:

```json
{"name": "Thai License Plate Detector", "author": "TH Support Cytron (Roboflow workspace th-support-cytron-yvkeg)",
 "url": "https://universe.roboflow.com/th-support-cytron-yvkeg/thai-license-plate-detector",
 "licence": "CC BY 4.0", "licence_url": "https://creativecommons.org/licenses/by/4.0/",
 "checked_by": "your name", "checked_on": "2026-11-20", "notes": "licence seen on the page and in README.roboflow.txt"}
```

A folder without `SOURCE.json` is **skipped** (no unverified data enters training).

In [ ]:
# 2. DATA IN — mount Google Drive (or upload a zip to /content/raw)
USE_DRIVE = True
RAW = "/content/drive/MyDrive/lensiq-plate-data" if USE_DRIVE else "/content/raw"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
sources = sorted(d for d in glob.glob(os.path.join(RAW, "*")) if os.path.isdir(d))
ok_sources = []
for d in sources:
    meta = os.path.join(d, "SOURCE.json")
    if not os.path.exists(meta):
        print("SKIP (no SOURCE.json):", d); continue
    m = json.load(open(meta, encoding="utf-8"))
    bad = [k for k in ("name", "url", "licence", "checked_by") if not m.get(k)]
    if bad or any(x in m["licence"].upper() for x in ("NC", "ND", "AGPL", "GPL")):
        print("SKIP (missing", bad, "or excluded licence", m.get("licence"), "):", d); continue
    m["dir"] = d; ok_sources.append(m)
print(len(ok_sources), "verified sources:", [s["name"] for s in ok_sources])

In [ ]:
# 3. MERGE + DEDUPE + ONE CLASS 'plate' — reads COCO json / YOLO txt / VOC xml, every box becomes class 0 'plate'
import numpy as np
from PIL import Image
import xml.etree.ElementTree as ET
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
PLATE_WORDS = ("plate", "license", "licence", "lp", "ป้าย", "ทะเบียน")   # class names kept; character-level classes are dropped

def is_plate(name):
    n = str(name).lower()
    return any(w in n for w in PLATE_WORDS)

def from_coco(js_path):
    js = json.load(open(js_path, encoding="utf-8")); base = os.path.dirname(js_path)
    cats = {c["id"]: c["name"] for c in js.get("categories", [])}
    keep = {k for k, v in cats.items() if is_plate(v)} or set(cats)   # single-class exports name it e.g. "License-Plate"
    imgs = {i["id"]: i for i in js["images"]}; out = {}
    for a in js["annotations"]:
        if a["category_id"] not in keep: continue
        im = imgs[a["image_id"]]; x, y, w, h = a["bbox"]
        out.setdefault(os.path.join(base, im["file_name"]), []).append([x, y, x + w, y + h])
    for i in imgs.values(): out.setdefault(os.path.join(base, i["file_name"]), [])
    return out

def from_yolo(img_path):
    lab = os.path.splitext(img_path.replace("/images/", "/labels/"))[0] + ".txt"
    if not os.path.exists(lab): return None
    W, H = Image.open(img_path).size; boxes = []
    for line in open(lab):
        p = line.split()
        if len(p) < 5: continue
        cx, cy, w, h = map(float, p[1:5]); boxes.append([(cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H])
    return boxes

def from_voc(img_path):
    x = os.path.splitext(img_path)[0] + ".xml"
    if not os.path.exists(x): return None
    boxes = []
    for o in ET.parse(x).getroot().iter("object"):
        if not is_plate(o.findtext("name", "plate")): continue
        b = o.find("bndbox"); boxes.append([float(b.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax")])
    return boxes

def dhash(path, n=8):   # perceptual difference hash: Roboflow forks often contain the same photos
    im = Image.open(path).convert("L").resize((n + 1, n), Image.BILINEAR); a = np.asarray(im, dtype=np.int16)
    return int("".join("1" if v else "0" for v in (a[:, 1:] > a[:, :-1]).flatten()), 2)

records, seen = [], {}
for s in ok_sources:
    items = {}
    for js in glob.glob(os.path.join(s["dir"], "**", "*.json"), recursive=True):
        if os.path.basename(js) == "SOURCE.json": continue
        try: items.update(from_coco(js))
        except Exception as e: print("not COCO:", js, e)
    for p in glob.glob(os.path.join(s["dir"], "**", "*"), recursive=True):
        if not p.lower().endswith(IMG_EXT) or p in items: continue
        b = from_yolo(p)
        if b is None: b = from_voc(p)
        if b is not None: items[p] = b
    dup = 0
    for p, boxes in items.items():
        if not os.path.exists(p): continue
        h = dhash(p)
        if any(bin(h ^ k).count("1") <= 4 for k in seen):   # near-duplicate of an image already taken (any source)
            dup += 1; continue
        seen[h] = p
        records.append({"file": p, "source": s["name"], "boxes": [b for b in boxes if b[2] - b[0] >= 4 and b[3] - b[1] >= 3], "hash": h})
    print(f"{s['name']}: {len(items)} images, {dup} near-duplicates dropped")
print(len(records), "images,", sum(len(r["boxes"]) for r in records), "plate boxes")

In [ ]:
# 4. SPLIT BY SOURCE — 80/10/10 inside every source (seeded); HOLDOUT_SOURCES go entirely to test (cross-source check)
SEED = 20261108
HOLDOUT_SOURCES = []          # e.g. ["License Plate (yas-kk2tk)"] — a whole source never seen in training
random.seed(SEED)
split = {"train": [], "val": [], "test": []}
for name in sorted({r["source"] for r in records}):
    rs = sorted([r for r in records if r["source"] == name], key=lambda r: r["hash"]); random.shuffle(rs)
    if name in HOLDOUT_SOURCES: split["test"] += rs; continue
    n = len(rs); a, b = int(n * 0.8), int(n * 0.9)
    split["train"] += rs[:a]; split["val"] += rs[a:b]; split["test"] += rs[b:]
DS = os.path.join(WORK, "plates_coco"); shutil.rmtree(DS, ignore_errors=True)
for part, rs in split.items():
    folder = part + "2017"   # YOLOX 0.3.0 COCODataset reads images from <data_dir>/<name>, name = train2017 / val2017 / test2017
    os.makedirs(os.path.join(DS, folder), exist_ok=True)
    images, anns = [], []
    for i, r in enumerate(rs):
        ext = os.path.splitext(r["file"])[1].lower(); fn = f"{part}_{i:06d}{ext}"
        shutil.copy(r["file"], os.path.join(DS, folder, fn)); W, H = Image.open(r["file"]).size
        images.append({"id": i, "file_name": fn, "width": W, "height": H, "source": r["source"], "orig": os.path.relpath(r["file"], RAW)})
        for b in r["boxes"]:
            anns.append({"id": len(anns), "image_id": i, "category_id": 1, "bbox": [b[0], b[1], b[2] - b[0], b[3] - b[1]], "area": (b[2] - b[0]) * (b[3] - b[1]), "iscrowd": 0})
    os.makedirs(os.path.join(DS, "annotations"), exist_ok=True)
    json.dump({"images": images, "annotations": anns, "categories": [{"id": 1, "name": "plate"}]}, open(os.path.join(DS, "annotations", f"{part}.json"), "w"))
    print(part, len(images), "images", len(anns), "boxes")

## 5. Conditions CSV (fill it in for the TEST images)

The cell writes `conditions.csv` with one row per test image. Fill the columns by looking at the photos:
`colour` = white / yellow / green / red / blue / graphic / other, `motorcycle` = 1/0, `night` = 1/0. Upload it back
(same path). Per-condition recall in step 9 uses it; empty cells are reported as "untagged".

In [ ]:
# 5. CONDITIONS TEMPLATE
COND = os.path.join(WORK, "conditions.csv")
if not os.path.exists(COND):
    te = json.load(open(os.path.join(DS, "annotations", "test.json")))
    with open(COND, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["file_name", "source", "orig", "colour", "motorcycle", "night"])
        for im in te["images"]: w.writerow([im["file_name"], im["source"], im["orig"], "", "", ""])
    print("wrote", COND, "- download, fill, upload back, then re-run step 9")
else:
    print("using", COND)

In [ ]:
# 6. YOLOX EXP — YOLOX-nano, 1 class, 416 input (set INPUT = 320 for a faster model), night / colour augmentation
INPUT = 416
EPOCHS = 150
NIGHT_COPIES = 0.3            # share of training images duplicated as simulated night (dark, noisy, blurred, colour-shifted)
def night_copy(src, dst, rnd):
    im = np.asarray(Image.open(src).convert("RGB"), dtype=np.float32) / 255
    im = np.power(im, rnd.uniform(1.8, 3.0)) * rnd.uniform(0.35, 0.7)              # dark, gamma
    im = im * np.array([rnd.uniform(0.8, 1.1), rnd.uniform(0.8, 1.05), rnd.uniform(0.7, 1.2)])  # sodium / LED colour cast
    im = im + np.random.default_rng(rnd.randint(0, 1 << 30)).normal(0, rnd.uniform(0.01, 0.04), im.shape)
    out = Image.fromarray((np.clip(im, 0, 1) * 255).astype(np.uint8))
    if rnd.random() < 0.5:
        from PIL import ImageFilter; out = out.filter(ImageFilter.GaussianBlur(rnd.uniform(0.5, 1.5)))
    out.save(dst, quality=92)
tr = json.load(open(os.path.join(DS, "annotations", "train.json"))); rnd = random.Random(SEED); extra = []
for im in list(tr["images"]):
    if rnd.random() >= NIGHT_COPIES: continue
    nid = len(tr["images"]) + len(extra); fn = "night_" + im["file_name"]
    night_copy(os.path.join(DS, "train2017", im["file_name"]), os.path.join(DS, "train2017", fn), rnd)
    extra.append({**im, "id": nid, "file_name": fn, "night_sim": True})
    for a in [a for a in tr["annotations"] if a["image_id"] == im["id"]]:
        tr["annotations"].append({**a, "id": len(tr["annotations"]), "image_id": nid})
tr["images"] += extra; json.dump(tr, open(os.path.join(DS, "annotations", "train.json"), "w"))
print("night copies:", len(extra))
EXP = f'''
import os
from yolox.exp import Exp as MyExp
class Exp(MyExp):
    def __init__(self):
        super().__init__()
        self.num_classes = 1
        self.depth = 0.33; self.width = 0.25                    # YOLOX-nano
        self.input_size = ({INPUT}, {INPUT}); self.test_size = ({INPUT}, {INPUT})
        self.random_size = (10, 20); self.mosaic_scale = (0.5, 1.5)
        self.mosaic_prob = 0.5; self.enable_mixup = False; self.mixup_prob = 0.0   # nano recipe
        self.hsv_prob = 1.0; self.flip_prob = 0.0                # colour jitter on; no horizontal flip (text is not symmetric)
        self.degrees = 5.0; self.translate = 0.1; self.shear = 2.0
        self.max_epoch = {EPOCHS}; self.no_aug_epochs = 15; self.warmup_epochs = 5
        self.basic_lr_per_img = 0.01 / 64.0; self.eval_interval = 5
        self.data_dir = "{DS}"
        self.train_ann = "train.json"; self.val_ann = "val.json"; self.test_ann = "test.json"
        self.exp_name = "plate_nano"
    def get_model(self, sublinear=False):
        import torch.nn as nn
        from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead
        def init_yolo(M):
            for m in M.modules():
                if isinstance(m, nn.BatchNorm2d): m.eps = 1e-3; m.momentum = 0.03
        if getattr(self, "model", None) is None:
            in_channels = [256, 512, 1024]
            backbone = YOLOPAFPN(self.depth, self.width, in_channels=in_channels, depthwise=True)
            head = YOLOXHead(self.num_classes, self.width, in_channels=in_channels, depthwise=True)
            self.model = YOLOX(backbone, head)
        self.model.apply(init_yolo); self.model.head.initialize_biases(1e-2)
        return self.model
    # No get_dataset / get_eval_dataset overrides: YOLOX 0.3.0's own get_data_loader / get_eval_loader build COCODataset
    # from data_dir + train_ann ("train2017") / val_ann ("val2017") / test_ann ("test2017" with --test), which is exactly the
    # layout written in step 4 (those two hooks only exist on YOLOX main after 0.3.0).
'''
open(os.path.join(WORK, "plate_nano_exp.py"), "w").write(EXP); print("exp written")

In [ ]:
# 7. TRAIN (several hours on a Colab T4). Resume with: --resume -c YOLOX_outputs/plate_nano/latest_ckpt.pth
os.chdir(os.path.join(WORK, "YOLOX"))
subprocess.run([sys.executable, "tools/train.py", "-f", os.path.join(WORK, "plate_nano_exp.py"), "-d", "1", "-b", "32",
                "--fp16", "-o", "-c", os.path.join(WORK, "yolox_nano.pth")], check=True)
BEST = os.path.join(WORK, "YOLOX", "YOLOX_outputs", "plate_nano", "best_ckpt.pth"); print(BEST, os.path.exists(BEST))

In [ ]:
# 8. EXPORT ONNX — raw head output [1, A, 6] (no decode in graph), opset 13: runs on onnxruntime-web WASM and is decoded by
#    LQDetect.decodeYolox (lensiq-detect.js) like the vendored person model
os.chdir(os.path.join(WORK, "YOLOX"))
ONNX = os.path.join(WORK, f"plate_yolox_nano_{INPUT}.onnx")
subprocess.run([sys.executable, "tools/export_onnx.py", "--output-name", ONNX, "-f", os.path.join(WORK, "plate_nano_exp.py"),
                "-c", BEST, "--opset", "13", "--no-onnxsim"], check=True)
import onnx; m = onnx.load(ONNX); onnx.checker.check_model(m)
print("ops:", sorted({n.op_type for n in m.graph.node}))
print("in:", [(i.name, [d.dim_value for d in i.type.tensor_type.shape.dim]) for i in m.graph.input])
print("out:", [(o.name, [d.dim_value for d in o.type.tensor_type.shape.dim]) for o in m.graph.output])
QUANTIZE = False   # LENSIQ measured LPD-YuNet int8 ~6x SLOWER than fp32 on onnxruntime-web WASM: measure before shipping int8
if QUANTIZE:
    from onnxruntime.quantization import quantize_dynamic, QuantType
    quantize_dynamic(ONNX, ONNX.replace(".onnx", "_int8.onnx"), weight_type=QuantType.QUInt8)
print(ONNX, round(os.path.getsize(ONNX) / 1e6, 2), "MB")

In [ ]:
# 9. EVALUATE — COCO mAP (YOLOX eval on the test split) + per-condition recall at IoU >= 0.5 with the exported ONNX
os.chdir(os.path.join(WORK, "YOLOX"))
r = subprocess.run([sys.executable, "tools/eval.py", "-f", os.path.join(WORK, "plate_nano_exp.py"), "-c", BEST, "-b", "32", "-d", "1",
                    "--conf", "0.001", "--test"], capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:]); EVAL_LOG = r.stdout[-3000:]
import onnxruntime as ort
sess = ort.InferenceSession(ONNX, providers=["CPUExecutionProvider"])
def letterbox(img, S):
    W, H = img.size; k = min(S / W, S / H); im = img.resize((int(W * k), int(H * k)))
    c = Image.new("RGB", (S, S), (114, 114, 114)); c.paste(im, (0, 0))
    return np.asarray(c)[:, :, ::-1].transpose(2, 0, 1)[None].astype(np.float32), k   # BGR 0..255 like the vendored YOLOX
def decode(out, S, k, conf=0.3, iou=0.5):
    grids, strides = [], []
    for s in (8, 16, 32):
        n = S // s; yv, xv = np.meshgrid(np.arange(n), np.arange(n), indexing="ij")
        grids.append(np.stack((xv, yv), 2).reshape(-1, 2)); strides.append(np.full((n * n, 1), s))
    g = np.concatenate(grids); st = np.concatenate(strides); o = out[0]
    xy = (o[:, :2] + g) * st; wh = np.exp(o[:, 2:4]) * st; sc = o[:, 4] * o[:, 5]
    keep = sc >= conf; b = np.concatenate([xy - wh / 2, xy + wh / 2], 1)[keep] / k; sc = sc[keep]
    order = sc.argsort()[::-1]; out_b = []
    while order.size:
        i = order[0]; out_b.append(b[i])
        xx1 = np.maximum(b[i, 0], b[order[1:], 0]); yy1 = np.maximum(b[i, 1], b[order[1:], 1])
        xx2 = np.minimum(b[i, 2], b[order[1:], 2]); yy2 = np.minimum(b[i, 3], b[order[1:], 3])
        inter = np.clip(xx2 - xx1, 0, None) * np.clip(yy2 - yy1, 0, None)
        a = (b[i, 2] - b[i, 0]) * (b[i, 3] - b[i, 1]); bb = (b[order[1:], 2] - b[order[1:], 0]) * (b[order[1:], 3] - b[order[1:], 1])
        order = order[1:][inter / (a + bb - inter + 1e-9) < iou]
    return out_b
def iou1(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1])); i = ix * iy
    return i / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - i + 1e-9)
te = json.load(open(os.path.join(DS, "annotations", "test.json")))
tags = {row["file_name"]: row for row in csv.DictReader(open(COND, encoding="utf-8"))} if os.path.exists(COND) else {}
gt = {}
for a in te["annotations"]: x, y, w, h = a["bbox"]; gt.setdefault(a["image_id"], []).append([x, y, x + w, y + h])
groups = {}
def add(group, hit): g = groups.setdefault(group, [0, 0]); g[0] += hit; g[1] += 1
ms = []
for im in te["images"]:
    img = Image.open(os.path.join(DS, "test2017", im["file_name"])).convert("RGB"); x, k = letterbox(img, INPUT)
    t0 = time.time(); dets = decode(sess.run(None, {sess.get_inputs()[0].name: x})[0], INPUT, k); ms.append((time.time() - t0) * 1000)
    t = tags.get(im["file_name"], {})
    for g in gt.get(im["id"], []):
        hit = int(any(iou1(d, g) >= 0.5 for d in dets))
        add("all", hit); add("source:" + im["source"], hit)
        add("colour:" + (t.get("colour") or "untagged"), hit)
        add("motorcycle" if t.get("motorcycle") == "1" else "car/untagged", hit)
        add("night" if t.get("night") == "1" else "day/untagged", hit)
RECALL = {k: {"hit": v[0], "n": v[1], "recall": round(v[0] / v[1], 3)} for k, v in sorted(groups.items())}
for k, v in RECALL.items(): print(f"{k:40s} {v['hit']:4d}/{v['n']:<4d} {v['recall']:.3f}")
print("CPU ms/image (median):", round(float(np.median(ms)), 1))

In [ ]:
# 10. MODEL CARD — dataset attributions (CC BY 4.0 needs title, author, source, licence + link, changes made)
card = [f"# Thai licence-plate detector — YOLOX-nano {INPUT} (LENSIQ Gate)", "",
        f"- File: `{os.path.basename(ONNX)}` ({round(os.path.getsize(ONNX) / 1e6, 2)} MB), one class `plate`, raw YOLOX head output, opset 13",
        f"- Trainer: Megvii YOLOX {YOLOX_TAG} (commit `{YOLOX_COMMIT}`), Apache-2.0. Start weights: YOLOX-nano COCO (release 0.1.1rc0), Apache-2.0",
        f"- Training: {EPOCHS} epochs, input {INPUT}, mosaic 0.5, HSV jitter, ±5° rotation, simulated-night copies {NIGHT_COPIES:.0%} of train, seed {SEED}",
        "- Not used: Ultralytics (AGPL-3.0), WongKinYiu / open-image-models YOLOv9 weights (GPL-derived, unclear data) — not as start weights, not for labels", "",
        "## Data (images are NOT distributed; only these weights)", "",
        "| Dataset | Author / owner | Source | Licence | Images used | Changes |", "|---|---|---|---|---|---|"]
for s in ok_sources:
    n = sum(1 for r in records if r["source"] == s["name"])
    card.append(f"| {s['name']} | {s.get('author', '')} | {s['url']} | [{s['licence']}]({s.get('licence_url', '')}) | {n} | merged into one class 'plate', near-duplicates removed, re-split, used to train a model |")
card += ["", "Licences were checked by hand on each dataset page and in its export README (see SOURCE.json: checked_by / checked_on).", "",
         "## Results (held-out test split — synthetic or web photos are NOT a field measurement)", "", "```", EVAL_LOG.strip(), "```", "",
         "| Group | Hit / n | Recall @ IoU 0.5 |", "|---|---|---|"]
card += [f"| {k} | {v['hit']}/{v['n']} | {v['recall']:.3f} |" for k, v in RECALL.items()]
card += ["", "## Intended use and limits", "",
         "Suggests where the plate is for LENSIQ Gate; a person confirms every read. Plates are personal data (PDPA): pilot photos",
         "need consent / signage and a retention limit. Small data: expect lower recall at night, on motorcycles, coloured or",
         "graphic plates and on cameras unlike the training photos — measure on site (docs/FIELD-PILOT.md) before relying on it."]
open(os.path.join(WORK, "MODEL_CARD.md"), "w", encoding="utf-8").write("\n".join(card)); print("\n".join(card[:12]))

In [ ]:
# 11. DOWNLOAD — the ONNX file and MODEL_CARD.md (copy into vendor/models/plate-det-th/ with a LICENSE + README, see docs/PLATE-DATASETS.md)
from google.colab import files
files.download(ONNX); files.download(os.path.join(WORK, "MODEL_CARD.md"))